# Superheterodyne Receiver Chain

Simulation of a basic superheterodyne receiver front-end combining multiple PathSim-RF blocks: an RF amplifier (LNA), a mixer for downconversion, and an IF amplifier. This demonstrates how the blocks compose into a complete signal chain.

## Receiver Architecture

The receiver chain consists of:
1. **LNA** (Low Noise Amplifier): 15 dB gain, IIP3 = +5 dBm
2. **Mixer**: Downconverts RF to IF with 0 dB conversion gain
3. **IF Amplifier**: 20 dB gain, IIP3 = +15 dBm

The RF input at 1000 Hz is downconverted to an IF of 100 Hz using a 900 Hz LO.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope, Spectrum
from pathsim.solvers import RKCK54

from pathsim_rf import RFAmplifier, RFMixer

## System Setup

In [ ]:
# Signal frequencies
f_rf = 1000.0
f_lo = 900.0
f_if = f_rf - f_lo

# Input signal (weak RF signal)
Z0 = 50.0
p_in_dbm = -30.0  # -30 dBm input
p_watts = 10.0 ** (p_in_dbm / 10.0) * 1e-3
v_rf = np.sqrt(2.0 * Z0 * p_watts)

# LO amplitude (strong local oscillator)
v_lo = 1.0  # 1V peak

# Build signal chain
rf_src = Source(func=lambda t: v_rf * np.sin(2 * np.pi * f_rf * t))
lo_src = Source(func=lambda t: v_lo * np.sin(2 * np.pi * f_lo * t))

lna = RFAmplifier(gain=15.0, IIP3=5.0, Z0=Z0)
mixer = RFMixer(conversion_gain=0.0, Z0=Z0)
if_amp = RFAmplifier(gain=20.0, IIP3=15.0, Z0=Z0)

# Observation points at each stage
sco_rf = Scope(labels=['RF input'])
sco_lna = Scope(labels=['LNA output'])
sco_if = Scope(labels=['IF output'])

# Spectrum at output
freq = np.linspace(0, 2500, 500)
spc = Spectrum(freq=freq, labels=['IF output'])

## Connections

The signal flows: RF Source -> LNA -> Mixer (RF port) -> IF Amplifier -> Output. The LO connects to the mixer's LO port.

In [ ]:
connections = [
    Connection(rf_src, lna, sco_rf),
    Connection(lo_src, mixer["lo"]),
    Connection(lna, mixer["rf"], sco_lna),
    Connection(mixer, if_amp),
    Connection(if_amp, sco_if, spc),
]

sim = Simulation(
    [rf_src, lo_src, lna, mixer, if_amp, sco_rf, sco_lna, sco_if, spc],
    connections,
    dt=1 / (20 * (f_rf + f_lo)),
    Solver=RKCK54,
    tolerance_lte_rel=1e-5,
    tolerance_lte_abs=1e-7
)

sim.run(20 / f_if)

## Signal at Each Stage

Let's visualize how the signal evolves through the receiver chain.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 7), dpi=120, sharex=True)

t, [y] = sco_rf.read()
axes[0].plot(np.array(t) * 1000, np.array(y) * 1000, linewidth=1)
axes[0].set_ylabel('Voltage [mV]')
axes[0].set_title(f'RF Input ({p_in_dbm:.0f} dBm at {f_rf:.0f} Hz)')
axes[0].grid(True, alpha=0.3)

t, [y] = sco_lna.read()
axes[1].plot(np.array(t) * 1000, np.array(y) * 1000, linewidth=1)
axes[1].set_ylabel('Voltage [mV]')
axes[1].set_title('After LNA (+15 dB)')
axes[1].grid(True, alpha=0.3)

t, [y] = sco_if.read()
axes[2].plot(np.array(t) * 1000, np.array(y) * 1000, linewidth=1)
axes[2].set_ylabel('Voltage [mV]')
axes[2].set_xlabel('Time [ms]')
axes[2].set_title('IF Output After Mixer + IF Amp (+20 dB)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Output Spectrum

The IF output spectrum shows the downconverted signal at $f_{\mathrm{IF}} = 100$ Hz along with the image at $f_{\mathrm{RF}} + f_{\mathrm{LO}} = 1900$ Hz.

In [ ]:
f, [G] = spc.read()

fig, ax = plt.subplots(dpi=120)
ax.plot(f, 20 * np.log10(np.abs(G) + 1e-15), linewidth=2)
ax.axvline(x=f_if, color='red', linestyle='--', alpha=0.5, label=f'IF = {f_if:.0f} Hz')
ax.axvline(x=f_rf + f_lo, color='orange', linestyle='--', alpha=0.5, label=f'Image = {f_rf + f_lo:.0f} Hz')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude [dB]')
ax.set_title('IF Output Spectrum')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()